## SECTION A

### Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing? 

### Limitations of MapReduce

**1. Disk I/O after every step**
MapReduce writes intermediate results to HDFS (disk) after every Map and Reduce phase.
So if your job has 10 steps, you're reading and writing to disk 10 times.
Disk I/O is slow — this is the biggest bottleneck.

**2. Not good for iterative algorithms**
Machine learning algorithms like k-means or gradient descent need to run
the same operation over data hundreds of times. With MapReduce, every
iteration means another round of disk reads and writes. It becomes painfully slow.

**3. No real-time processing**
MapReduce is batch-only. You cannot process a stream of live data with it.
There's no concept of "process this as it arrives."

**4. High latency**
Even a simple query takes a long time because of the overhead of spinning up
Map tasks, shuffling data, writing to disk, and starting Reduce tasks.
Not suitable for interactive data exploration at all.

**5. Complex to write**
Even simple operations need to be manually coded as Mapper and Reducer classes.
No built-in SQL support, no DataFrame abstraction.

---

### Why Spark is better

| Limitation in MapReduce | How Spark fixes it |
|---|---|
| Disk I/O after every step | Keeps data in memory (RAM) across steps |
| Slow iterative algorithms | In-memory caching makes repeated passes fast |
| No real-time processing | Spark Streaming handles live data |
| High latency | Runs 10x–100x faster than MapReduce in benchmarks |
| Complex to write | DataFrame API, SQL support, Python/Scala friendly |

Spark doesn't completely replace disk — it falls back to disk when data
doesn't fit in memory. But for most workloads, in-memory processing
makes an enormous difference in speed.

----------------------------- 

### Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Machine learning algorithms are iterative by nature.
Take gradient descent for example — it runs the same computation 
over the entire dataset hundreds of times, adjusting weights each pass.

With MapReduce (disk-based):
- Iteration 1: read from disk → compute → write result to disk
- Iteration 2: read from disk → compute → write result to disk
- Iteration 3: read from disk → compute → write result to disk
- ... repeat N times

Every single iteration hits the disk. Disk read/write speed is slower than RAM. So N iterations = N full disk reads.

---

### How Spark handles it differently

Spark introduced the concept of RDDs (Resilient Distributed Datasets)
and later DataFrames — both can be persisted in memory using .cache() or .persist().

With Spark:
- Load data into memory once
- Iteration 1: compute in RAM → keep result in RAM
- Iteration 2: compute in RAM → keep result in RAM
- ... repeat 100 times, all in memory

The data never touches disk between iterations unless memory runs out.

---

### Real example  k-means clustering

k-means repeatedly scans the entire dataset to reassign cluster labels.

| System | 10 iterations on 10GB data | Why |
|---|---|---|
| MapReduce | ~60 minutes | Reads 10GB from disk 10 times |
| Spark (cached) | ~2–3 minutes | Reads 10GB from RAM 10 times |

Spark's own benchmarks showed **100x speedup** over Hadoop MapReduce
on iterative ML workloads when data fits in memory.

---




### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5_Spark_DataFrames") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark version : {spark.version}")
print("SparkSession created successfully ✓")

C:\Users\abc\anaconda3\envs\mldev\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


PySpark version : 4.2.0
SparkSession created successfully ✓


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random, os

np.random.seed(42)
random.seed(42)

n = 1000
regions    = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Food', 'Sports', 'Home']
cities     = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai',
              'Pune', 'Kolkata', 'Ahmedabad', 'Jaipur', 'Surat']
subs       = ['Premium', 'Basic', 'Free']
store_ids  = [f'S{i:03d}' for i in range(1, 6)]
base       = datetime(2024, 1, 1)

user_ids = [f'U{random.randint(1, 800):04d}' for _ in range(n)]
dates    = [(base + timedelta(days=random.randint(0, 364))).strftime('%Y-%m-%d')
            for _ in range(n)]

# Inject 60 duplicate (user_id, date) pairs deliberately for Q3
for _ in range(60):
    i, j = random.randint(0, n-1), random.randint(0, n-1)
    user_ids[j], dates[j] = user_ids[i], dates[i]

data = {
    'user_id'          : user_ids,
    'transaction_date' : dates,
    'region'           : [random.choice(regions)    for _ in range(n)],
    'product_category' : [random.choice(categories) for _ in range(n)],
    'sale_amount'      : [round(random.uniform(100, 10000), 2) for _ in range(n)],
    'city'             : [random.choice(cities)      for _ in range(n)],
    'age'              : [random.randint(15, 65)     for _ in range(n)],
    'subscription'     : [random.choice(subs)        for _ in range(n)],
    'email'            : [f'user{i}@email.com' if random.random() > 0.10 else None
                          for i in range(n)],
    'username'         : [f'user_{i}' if random.random() > 0.08 else ''
                          for i in range(n)],
    'status'           : [random.choice(['Active','Inactive','Pending'])
                          if random.random() > 0.12 else None
                          for _ in range(n)],
    'price'            : [round(random.uniform(10, 5000), 2)
                          if random.random() > 0.08 else None
                          for _ in range(n)],
    'store_id'         : [random.choice(store_ids)  for _ in range(n)],
    'raw_timestamp'    : [(base + timedelta(days=random.randint(0, 364),
                           hours=random.randint(0, 23),
                           minutes=random.randint(0, 59))).strftime('%Y-%m-%d %H:%M:%S')
                          for _ in range(n)],
}

df_pd = pd.DataFrame(data)
os.makedirs('raw_data', exist_ok=True)
df_pd.to_csv('raw_data/ecommerce_data.csv', index=False)

print(f"✓ Dataset created: {len(df_pd)} rows × {len(df_pd.columns)} columns")
print(f"\nNull counts:")
print(df_pd.isnull().sum()[df_pd.isnull().sum() > 0])
print(f"\nEmpty username rows : {(df_pd['username'] == '').sum()}")

✓ Dataset created: 1000 rows × 14 columns

Null counts:
email      93
status    103
price      59
dtype: int64

Empty username rows : 79


In [3]:
df = spark.read.csv(
    'raw_data/ecommerce_data.csv',
    header=True,
    inferSchema=True
)

print(f"✓ Loaded: {df.count()} rows × {len(df.columns)} columns")
df.printSchema()
df.show(5, truncate=False)

✓ Loaded: 1000 rows × 14 columns
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)

+-------+----------------+------+----------------+-----------+---------+---+------------+---------------+--------+--------+-------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|city     |age|subscription|email          |username|status  |price  |store_id|raw_timestamp      |
+-------+----------------+------+----------------+-----------+-------

In [4]:
print(f"Rows before deduplication : {df.count()}")

df_deduped = df.dropDuplicates(['user_id', 'transaction_date'])

print(f"Rows after deduplication  : {df_deduped.count()}")
print(f"Duplicate rows removed    : {df.count() - df_deduped.count()}")

Rows before deduplication : 1000
Rows after deduplication  : 942
Duplicate rows removed    : 58


### Q4. Filter for rows where region is 'West', then group by product_category to find the average sale_amount.

In [5]:
from pyspark.sql.functions import avg, round as spark_round

df_sales = df

df_result = df_sales \
    .filter(df_sales['region'] == 'West') \
    .groupBy('product_category') \
    .agg(spark_round(avg('sale_amount'), 2).alias('avg_sale_amount')) \
    .orderBy('avg_sale_amount', ascending=False)

print(f"West region rows: {df_sales.filter(df_sales['region'] == 'West').count()}")
print("\nAverage sale amount by product category (West region):")
df_result.show(truncate=False)

West region rows: 267

Average sale amount by product category (West region):
+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|Clothing        |5415.96        |
|Food            |5180.85        |
|Home            |5062.73        |
|Sports          |4900.9         |
|Electronics     |4683.24        |
+----------------+---------------+



### Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'. 

**Key difference between .na.drop() and .na.fill():**

| | .na.drop() | .na.fill() |
|---|---|---|
| What it does | Removes the entire row | Replaces null with a value |
| Data loss | Yes — row is gone | No — row is kept |
| Use when | Row is unusable without that field | Missing value has a sensible default |
| Output rows | Fewer rows | Same number of rows |

**When to use which:**

.na.drop() — use when the null column is critical and the row
has no meaning without it. Example: dropping rows where order_id
is null because an order without an ID can't be processed.

.na.fill() — use when null just means "not provided" or "not applicable"
and you can substitute a safe default. Example: filling status with
'Unknown' means the record is still valid, just unclassified.

**In our case:**
status being null doesn't make the transaction invalid —
the sale happened, we just don't know the account status.
So .na.fill('Unknown') is the right call here, not .na.drop().

In [6]:
from pyspark.sql.functions import col, count, when

print("Null counts BEFORE handling:")
df.select([count(when(col(c).isNull(), c)).alias(c) 
           for c in ['status', 'email', 'price']]).show()

# --- .na.drop() --- drops entire row if null found in specified columns
df_dropped = df.na.drop(subset=['status'])
print(f"Rows using .na.drop() on status  : {df_dropped.count()}")

# --- .na.fill() --- replaces null with a given value, row is kept
df_filled = df.na.fill({'status': 'Unknown'})
print(f"Rows using .na.fill() on status  : {df_filled.count()}")

print("\nNull counts AFTER .na.fill():")
df_filled.select([count(when(col(c).isNull(), c)).alias(c)
                  for c in ['status', 'email', 'price']]).show()

print("\nSample status values after fill:")
df_filled.select('status').distinct().show()

Null counts BEFORE handling:
+------+-----+-----+
|status|email|price|
+------+-----+-----+
|   103|   93|   59|
+------+-----+-----+

Rows using .na.drop() on status  : 897
Rows using .na.fill() on status  : 1000

Null counts AFTER .na.fill():
+------+-----+-----+
|status|email|price|
+------+-----+-----+
|     0|   93|   59|
+------+-----+-----+


Sample status values after fill:
+--------+
|  status|
+--------+
| Unknown|
|  Active|
|Inactive|
| Pending|
+--------+



### Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 

In [7]:
from pyspark.sql.functions import count

df_city_count = df \
    .groupBy('city') \
    .agg(count('*').alias('total_records')) \
    .filter(count('*') > 100) \
    .orderBy('total_records', ascending=False)

df_city_count.show(truncate=False)
print(f"Cities with more than 100 records: {df_city_count.count()}")

+---------+-------------+
|city     |total_records|
+---------+-------------+
|Bangalore|111          |
|Jaipur   |110          |
|Ahmedabad|102          |
+---------+-------------+

Cities with more than 100 records: 3


### Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them? 

Once a Spark DataFrame is created, it cannot be changed. Ever. No in-place edits, no modifying a column directly, and no deleting a row from the existing object. 

This is fundamentally different from pandas where you can do:
```python
df['column'] = new_values  # Modifies the existing DataFrame
```

In Spark, every "cleaning" operation returns a brand new DataFrame. The original remains untouched.

### Spark Transformation Examples

* **Dropping a column**
  ```python
  # Does NOT modify df. Returns a NEW DataFrame.
  df_clean = df.drop('unwanted_column')
  ```
* **Renaming a column**
  ```python
  # df remains unchanged. df_renamed has the new name.
  # df_renamed has the new name.
  df_renamed = df.withColumnRenamed('raw_timestamp', 'event_time')
  ```
* **Modifying a column value**
  ```python
  # df keeps original prices. df_updated has GST-adjusted prices.
  df_updated = df.withColumn('price', df['price'] * 1.18)
  ```

### Core Advantages

* **Fault tolerance**
  * Spark recomputes DataFrames from scratch using lineage tracking.
  * Failed nodes replay transformation steps instead of restoring backups.
* **Safe parallel processing**
  * Multiple tasks read the same DataFrame simultaneously without corruption risks.
  * Shared data security is guaranteed since the state is unchangeable.
* **Reliable lazy evaluation**
  * Spark builds a dependable Directed Acyclic Graph (DAG) before execution.
  * Every node in the DAG represents a fixed, immutable state.


### Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 

In [9]:
from pyspark.sql.functions import col

df_filtered = df.filter(
    (col('age').between(18, 30)) &
    (col('subscription') == 'Premium')
)

print(f"Total rows in dataset           : {df.count()}")
print(f"Rows after filter               : {df_filtered.count()}")
print("\nSample results:")
df_filtered.select('user_id', 'age', 'subscription', 'region', 'sale_amount') \
           .show(10, truncate=False)

Total rows in dataset           : 1000
Rows after filter               : 83

Sample results:
+-------+---+------------+------+-----------+
|user_id|age|subscription|region|sale_amount|
+-------+---+------------+------+-----------+
|U0026  |23 |Premium     |West  |2291.38    |
|U0760  |22 |Premium     |West  |2328.61    |
|U0693  |28 |Premium     |West  |4793.56    |
|U0559  |28 |Premium     |East  |1814.69    |
|U0605  |18 |Premium     |East  |2315.32    |
|U0368  |27 |Premium     |South |4100.57    |
|U0619  |22 |Premium     |East  |1485.3     |
|U0550  |22 |Premium     |South |4105.53    |
|U0678  |27 |Premium     |East  |6610.39    |
|U0380  |22 |Premium     |West  |8727.18    |
+-------+---+------------+------+-----------+
only showing top 10 rows


### Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

Spark silently ignores nulls in aggregations.
That sounds helpful — but it leads to results that are 
technically computed but quietly wrong.

Prevent Skewed Ratios: Aggregate functions like avg() ignore NULL values in the denominator. If NULL values represent zeros, ignoring them skews the computed mean upward.

Join and Pipeline Integrity: Leaving NULL values unhandled can cause unexpected drops during join conditions or runtime exceptions in downstream analytical modules.

Explicit Data Logic: Replacing missing financial fields with 0 ensures business calculations remain accurate across all reporting layers.

### The right approach

Handle nulls first, then aggregate.
You decide what null means — don't let Spark decide silently.

**Option 1 — Fill nulls with 0 (null means no sale)**
df_clean = df.na.fill({'price': 0})
df_clean.agg(avg('price')).show()

**Option 2 — Drop null rows (null means unusable record)**
df_clean = df.na.drop(subset=['price'])
df_clean.agg(avg('price')).show()

**Option 3 — Fill with column mean (imputation)**
from pyspark.sql.functions import avg
mean_price = df.agg(avg('price')).collect()[0][0]
df_clean = df.na.fill({'price': round(mean_price, 2)})

The choice between these depends on the business context —
but making the choice explicitly and intentionally is always
better than letting Spark silently skip rows for you.

In [12]:
from pyspark.sql.functions import avg, count, col, when

null_count = df.filter(col('price').isNull()).count()
total      = df.count()
print(f"Total rows     : {total}")
print(f"Null prices    : {null_count}")
print(f"Non-null prices: {total - null_count}")

avg_before = df.agg(avg('price').alias('avg_price_before')).collect()[0][0]
print(f"\navg(price) BEFORE null handling : {round(avg_before, 2)}")
print(f"  → computed on {total - null_count} rows silently")

df_filled  = df.na.fill({'price': 0})
avg_filled = df_filled.agg(avg('price').alias('avg_price_filled')).collect()[0][0]
print(f"\navg(price) AFTER fill nulls=0   : {round(avg_filled, 2)}")
print(f"  → computed on all {total} rows")

df_dropped  = df.na.drop(subset=['price'])
avg_dropped = df_dropped.agg(avg('price').alias('avg_price_dropped')).collect()[0][0]
print(f"\navg(price) AFTER drop nulls     : {round(avg_dropped, 2)}")
print(f"  → computed on {df_dropped.count()} rows explicitly")

print(f"\nDifference between fill=0 and drop: {abs(round(avg_filled - avg_dropped, 2))}")
print("Same question, three different answers — null handling choice matters.")

Total rows     : 1000
Null prices    : 59
Non-null prices: 941

avg(price) BEFORE null handling : 2408.81
  → computed on 941 rows silently

avg(price) AFTER fill nulls=0   : 2266.69
  → computed on all 1000 rows

avg(price) AFTER drop nulls     : 2408.81
  → computed on 941 rows explicitly

Difference between fill=0 and drop: 142.12
Same question, three different answers — null handling choice matters.


### Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time. 

In [13]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import TimestampType

# Check current dtype of raw_timestamp before casting
print("Schema BEFORE:")
df.select('raw_timestamp').printSchema()
df.select('raw_timestamp').show(3, truncate=False)

# Step 1: Cast to TimestampType
# Step 2: Rename to event_time
df_revised = df \
    .withColumn('raw_timestamp', col('raw_timestamp').cast(TimestampType())) \
    .withColumnRenamed('raw_timestamp', 'event_time')

# Verify
print("Schema AFTER:")
df_revised.select('event_time').printSchema()
df_revised.select('event_time').show(3, truncate=False)

Schema BEFORE:
root
 |-- raw_timestamp: timestamp (nullable = true)

+-------------------+
|raw_timestamp      |
+-------------------+
|2024-11-15 09:43:00|
|2024-07-20 04:07:00|
|2024-12-08 06:40:00|
+-------------------+
only showing top 3 rows
Schema AFTER:
root
 |-- event_time: timestamp (nullable = true)

+-------------------+
|event_time         |
+-------------------+
|2024-11-15 09:43:00|
|2024-07-20 04:07:00|
|2024-12-08 06:40:00|
+-------------------+
only showing top 3 rows


### Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 


### Narrow vs Wide Transformations — the key distinction

In Spark, transformations fall into two categories:

**Narrow transformation:**
Each input partition contributes to exactly one output partition.
No data needs to move between machines.
Examples: filter(), select(), withColumn()

**Wide transformation:**
Each input partition may contribute to multiple output partitions.
Data must physically move across the network between executors.
Examples: groupBy(), join(), distinct(), orderBy()

groupBy() is a wide transformation — and Shuffle is why.

---


Shuffle is the process of:
1. Reading all partitions across all machines
2. Reorganizing rows so all rows with the same groupBy key
   end up on the same machine
3. Writing these reorganized rows to disk temporarily
4. Sending them over the network to the right executor
5. Each executor then aggregates its assigned keys locally

---

### Why Shuffle is expensive

| Cost | Why |
|---|---|
| Network I/O | Data physically moves between machines over the network |
| Disk I/O | Shuffle writes intermediate results to disk before sending |
| Memory pressure | Large shuffles can cause out-of-memory errors |
| Time | Can account for 70-80% of total job time on large datasets |

---

### Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string. 

In [14]:
from pyspark.sql.functions import col

# Step 1 — Identify problematic rows first
null_email    = df.filter(col('email').isNull()).count()
empty_username = df.filter(col('username') == '').count()
both_bad      = df.filter(
    col('email').isNull() | (col('username') == '')
).count()

print(f"Rows with null email          : {null_email}")
print(f"Rows with empty username      : {empty_username}")
print(f"Rows failing either condition : {both_bad}")
print(f"Total rows before cleaning    : {df.count()}")

# Step 2 — Remove rows where email is null OR username is empty
df_clean = df.filter(
    col('email').isNotNull() &
    (col('username') != '')
)

print(f"\nTotal rows after cleaning     : {df_clean.count()}")
print(f"Rows removed                  : {df.count() - df_clean.count()}")

# Step 3 — Verify no bad rows remain
remaining_null_email     = df_clean.filter(col('email').isNull()).count()
remaining_empty_username = df_clean.filter(col('username') == '').count()
print(f"\nVerification:")
print(f"Null emails remaining    : {remaining_null_email}")
print(f"Empty usernames remaining: {remaining_empty_username}")

Rows with null email          : 93
Rows with empty username      : 0
Rows failing either condition : 93
Total rows before cleaning    : 1000

Total rows after cleaning     : 833
Rows removed                  : 167

Verification:
Null emails remaining    : 0
Empty usernames remaining: 0


### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [15]:
from pyspark.sql.functions import min, max, avg, count, stddev, round as spark_round

# Single column — multiple stats at once
print("=== Price Column Statistics ===")
df.agg(
    spark_round(min('price'),    2).alias('min_price'),
    spark_round(max('price'),    2).alias('max_price'),
    spark_round(avg('price'),    2).alias('mean_price'),
    spark_round(stddev('price'), 2).alias('stddev_price'),
    count('price')                 .alias('non_null_count')
).show(truncate=False)

# Multiple columns — multiple stats at once
print("=== Sale Amount + Price Statistics by Region ===")
df.groupBy('region').agg(
    spark_round(min('sale_amount'),  2).alias('min_sale'),
    spark_round(max('sale_amount'),  2).alias('max_sale'),
    spark_round(avg('sale_amount'),  2).alias('avg_sale'),
    spark_round(avg('price'),        2).alias('avg_price'),
    count('*')                         .alias('total_records')
).orderBy('avg_sale', ascending=False).show(truncate=False)

=== Price Column Statistics ===
+---------+---------+----------+------------+--------------+
|min_price|max_price|mean_price|stddev_price|non_null_count|
+---------+---------+----------+------------+--------------+
|12.47    |4998.54  |2408.81   |1420.58     |941           |
+---------+---------+----------+------------+--------------+

=== Sale Amount + Price Statistics by Region ===
+------+--------+--------+--------+---------+-------------+
|region|min_sale|max_sale|avg_sale|avg_price|total_records|
+------+--------+--------+--------+---------+-------------+
|North |146.22  |9867.74 |5307.96 |2279.57  |232          |
|South |111.88  |9961.78 |5304.16 |2460.98  |243          |
|West  |165.57  |9948.06 |5069.65 |2450.51  |267          |
|East  |121.75  |9994.35 |5057.07 |2430.19  |258          |
+------+--------+--------+--------+---------+-------------+



### Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

### How inferSchema=True actually works

When you set inferSchema=True, Spark doesn't read the full dataset
to determine column types. It samples a small number of rows 
(by default the first 100 rows of each partition) and guesses
the schema based on what it sees.

This is fast — but it creates serious problems with messy date data.

---

### The risks

**Risk 1 — Spark samples only a subset of rows**
If the first 100 rows all have clean dates like '2024-01-15',
Spark infers TimestampType.
Then row 101 has '15/01/2024' — a different format.
Spark can't parse it as a timestamp → silently converts to null.
You lose data with no warning or error.

**Risk 2 — Mixed formats cause silent nulls**
Real-world data often has mixed date formats:
'2024-01-15'     ← ISO format
'15/01/2024'     ← European format
'Jan 15, 2024'   ← human-readable format
'20240115'       ← compact format

inferSchema picks one format based on what it samples first.
Everything else becomes null — silently.

**Risk 3 — Wrong type inference**
A column like '01-02-03' could be a date (Jan 2, 2003)
or a version number or a code. inferSchema guesses —
and it may guess wrong, giving you a timestamp column
full of meaningless values.

**Risk 4 — Extra scan over the data**
inferSchema requires Spark to read the data twice —
once to infer types, once to actually load it.
On large datasets (billions of rows), this doubles your load time
for a schema guess that might still be wrong.

---


#### Then handle dates explicitly with your own format
from pyspark.sql.functions import to_date
df = df.withColumn('transaction_date',
                   to_date(col('transaction_date'), 'yyyy-MM-dd'))

**Why this is better:**
- You control exactly what type each column gets
- No silent nulls from format mismatches
- Faster load — no schema inference scan
- Explicit date format means you catch bad data loudly, not silently

### Q15: Write a final processing pipeline that: 

Filters out duplicates. 

Fills null prices with 0. 

Groups by store_id to calculate total revenue. 

In [16]:
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, count

print("=" * 50)
print("PIPELINE STAGE 0 — Raw Data Snapshot")
print("=" * 50)
total_raw        = df.count()
duplicate_rows   = df.count() - df.dropDuplicates(['user_id','transaction_date']).count()
null_price_rows  = df.filter(col('price').isNull()).count()
print(f"Total rows         : {total_raw}")
print(f"Duplicate rows     : {duplicate_rows}")
print(f"Null price rows    : {null_price_rows}")

# ── STAGE 1: Remove duplicates ──────────────────────────────
print("\n" + "=" * 50)
print("PIPELINE STAGE 1 — Remove Duplicates")
print("=" * 50)
df_stage1 = df.dropDuplicates(['user_id', 'transaction_date'])
print(f"Rows after deduplication : {df_stage1.count()}")
print(f"Rows removed             : {total_raw - df_stage1.count()}")

# ── STAGE 2: Fill null prices with 0 ───────────────────────
print("\n" + "=" * 50)
print("PIPELINE STAGE 2 — Fill Null Prices")
print("=" * 50)
df_stage2 = df_stage1.na.fill({'price': 0})
remaining_nulls = df_stage2.filter(col('price').isNull()).count()
print(f"Null prices before fill  : {null_price_rows}")
print(f"Null prices after fill   : {remaining_nulls}")
print(f"Rows unchanged           : {df_stage2.count()}")

# ── STAGE 3: Group by store_id → total revenue ─────────────
print("\n" + "=" * 50)
print("PIPELINE STAGE 3 — Total Revenue by Store")
print("=" * 50)
df_stage3 = df_stage2 \
    .groupBy('store_id') \
    .agg(
        spark_round(spark_sum('price'), 2).alias('total_revenue'),
        count('*')                        .alias('total_transactions')
    ) \
    .orderBy('total_revenue', ascending=False)

df_stage3.show(truncate=False)

# ── FULL PIPELINE (chained, production style) ──────────────
print("=" * 50)
print("FULL PIPELINE — Chained Version")
print("=" * 50)
df_final = df \
    .dropDuplicates(['user_id', 'transaction_date']) \
    .na.fill({'price': 0}) \
    .groupBy('store_id') \
    .agg(
        spark_round(spark_sum('price'), 2).alias('total_revenue'),
        count('*')                        .alias('total_transactions')
    ) \
    .orderBy('total_revenue', ascending=False)

df_final.show(truncate=False)
print("✓ Pipeline complete")

PIPELINE STAGE 0 — Raw Data Snapshot
Total rows         : 1000
Duplicate rows     : 58
Null price rows    : 59

PIPELINE STAGE 1 — Remove Duplicates
Rows after deduplication : 942
Rows removed             : 58

PIPELINE STAGE 2 — Fill Null Prices
Null prices before fill  : 59
Null prices after fill   : 0
Rows unchanged           : 942

PIPELINE STAGE 3 — Total Revenue by Store
+--------+-------------+------------------+
|store_id|total_revenue|total_transactions|
+--------+-------------+------------------+
|S005    |438163.96    |192               |
|S002    |434608.32    |201               |
|S003    |433773.78    |179               |
|S004    |420533.61    |194               |
|S001    |404368.88    |176               |
+--------+-------------+------------------+

FULL PIPELINE — Chained Version
+--------+-------------+------------------+
|store_id|total_revenue|total_transactions|
+--------+-------------+------------------+
|S005    |438163.96    |192               |
|S002    |43460

**How the pipeline comes together:**

Stage 1 — dropDuplicates(['user_id', 'transaction_date'])
Removes rows where the same user made a transaction on the 
same date more than once.

Stage 2 — na.fill({'price': 0})
Null prices mean the price wasn't recorded — not that the 
item was free. But for revenue calculation, a null would be 
silently skipped by sum().

Stage 3 — groupBy('store_id').agg(sum('price'))
Groups all transactions by store, sums the price column
to get total revenue per store. 

---


**Full pipeline summary:**

| Stage | Operation | Rows In | Rows Out |
|---|---|---|---|
| Raw | Load CSV | — | 1000 |
| Stage 1 | dropDuplicates | 1000 | ~950 |
| Stage 2 | na.fill price=0 | ~950 | ~950 |
| Stage 3 | groupBy store_id | ~950 | 5 |

This is the standard shape of a Spark cleaning pipeline —
wide input, progressively cleaner, ends with an aggregated summary.